# 02 - Eksperimen & Komparasi 5 Model (*From Scratch* vs `scikit-learn`)
**Penanggung Jawab:** Kolektif Seluruh Anggota (Anggota 1 - 5)  
**Kelompok:** G15 - MessiLearning (IF3170 Inteligensi Artifisial)

---
Notebook ini berisi pengujian 5 algoritma Supervised Learning *from scratch* menggunakan data sintetis (selama fase paralel) maupun data bersih hasil `DataPipeline`, lengkap dengan komparasi metrik (Akurasi, Presisi, Recall, F1-Score, Waktu Eksekusi) terhadap `scikit-learn`.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.utils import (
    set_global_seed,
    RANDOM_SEED,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    calculate_custom_cost,
    k_fold_split,
)
from src.dummy_generator import generate_clean_synthetic_data, generate_dtl_synthetic_data
from src.knn import KNN
from src.dtl import DecisionTree, export_tree_visualization
from src.logistic_regression import LogisticRegression, generate_loss_contour_video
from src.svm import SVM, plot_svm_margin_2d
from src.naive_bayes import NaiveBayes, export_bayesian_network_graph

set_global_seed(RANDOM_SEED)
X_dummy, y_dummy = generate_clean_synthetic_data(n_samples=500, n_features=6)
X_dtl_dummy, y_dtl_dummy = generate_dtl_synthetic_data(n_samples=500)
print(f"Dummy Clean Shape: {X_dummy.shape}, Dummy DTL Shape: {X_dtl_dummy.shape}")

## 1. K-Nearest Neighbors (KNN) — Anggota 1
Komparasi `src.knn.KNN` vs `sklearn.neighbors.KNeighborsClassifier`

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# TODO (Anggota 1): Evaluasi KNN from scratch vs scikit-learn

## 2. Decision Tree Learning (DTL) — Anggota 2
Komparasi `src.dtl.DecisionTree` vs `sklearn.tree.DecisionTreeClassifier`, Visualisasi Top-$N$ Tree (+1 Poin Bonus), dan Optimasi Post-Pruning (+2 Poin Bonus)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from src.dtl import DecisionTree, export_tree_visualization, get_sklearn_dtl

root_dir = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
dtl_train_path = os.path.join(root_dir, 'data', 'processed', 'train_dtl.csv')

# 1. Pengujian pada Data Campuran Khusus DTL (Numerik + Kategorikal + Missing Values NaN)
if os.path.exists(dtl_train_path):
    df_dtl = pd.read_csv(dtl_train_path)
    target_col = 'is_canceled' if 'is_canceled' in df_dtl.columns else df_dtl.columns[-1]
    dtl_feat_names = [c for c in df_dtl.columns if c != target_col]
    X_dtl_eval = df_dtl[dtl_feat_names].to_numpy(dtype=object)
    y_dtl_eval = df_dtl[target_col].to_numpy(dtype=int)
    print(f"[DTL] Menggunakan dataset asli: {dtl_train_path} ({X_dtl_eval.shape})")
else:
    X_dtl_eval, _ = generate_dtl_synthetic_data(n_samples=500, random_state=RANDOM_SEED)
    rng_dtl = np.random.default_rng(RANDOM_SEED)
    num0 = np.where(pd.isna(X_dtl_eval[:, 0]), 50.0, X_dtl_eval[:, 0]).astype(float)
    num1 = np.where(pd.isna(X_dtl_eval[:, 1]), 50.0, X_dtl_eval[:, 1]).astype(float)
    cat2 = X_dtl_eval[:, 2]
    score_dtl = (
        (num0 > 48.0).astype(float) * 1.2
        + (num1 < 55.0).astype(float) * 0.9
        + np.isin(cat2, ['A', 'B']).astype(float) * 1.1
        + rng_dtl.normal(0, 0.45, size=len(X_dtl_eval))
    )
    y_dtl_eval = (score_dtl >= 1.6).astype(int)
    dtl_feat_names = ['lead_time', 'adr_price', 'market_segment', 'deposit_type']
    print(f"[DTL] Menggunakan dataset sintetis DTL (numerik + kategorikal + 5% NaN): {X_dtl_eval.shape}")

# Evaluasi 5-Fold Cross Validation untuk DTL From Scratch (ID3, C4.5, CART, & C4.5 + Post-Pruning REP)
dtl_configs = [
    ('DTL Scratch - ID3 (Entropy)', {'criterion': 'id3', 'use_pruning': False}),
    ('DTL Scratch - C4.5 (Gain Ratio)', {'criterion': 'c45', 'use_pruning': False}),
    ('DTL Scratch - CART (Gini)', {'criterion': 'cart', 'use_pruning': False}),
    ('DTL Scratch - C4.5 + REP Pruning (Bonus)', {'criterion': 'c45', 'use_pruning': True, 'pruning_method': 'rep', 'val_split_ratio': 0.2}),
    ('DTL Scratch - C4.5 + CCP Pruning (Bonus)', {'criterion': 'c45', 'use_pruning': True, 'pruning_method': 'ccp', 'ccp_alpha': 0.01}),
]

dtl_mixed_records = []
for name, params in dtl_configs:
    accs, precs, recs, f1s, costs, nodes_list, depths_list, times_ms = [], [], [], [], [], [], [], []
    for train_idx, val_idx in k_fold_split(X_dtl_eval, y_dtl_eval, k=5, seed=RANDOM_SEED):
        t0 = time.perf_counter()
        model = DecisionTree(**params, random_state=RANDOM_SEED)
        model.fit(X_dtl_eval[train_idx], y_dtl_eval[train_idx])
        preds = model.predict(X_dtl_eval[val_idx])
        elapsed = (time.perf_counter() - t0) * 1000.0
        accs.append(accuracy_score(y_dtl_eval[val_idx], preds))
        precs.append(precision_score(y_dtl_eval[val_idx], preds))
        recs.append(recall_score(y_dtl_eval[val_idx], preds))
        f1s.append(f1_score(y_dtl_eval[val_idx], preds))
        costs.append(calculate_custom_cost(y_dtl_eval[val_idx], preds))
        nodes_list.append(model.get_n_nodes())
        depths_list.append(model.get_depth())
        times_ms.append(elapsed)
    dtl_mixed_records.append({
        'Model (Mixed + NaN Dataset)': name,
        'Accuracy': np.mean(accs),
        'Precision': np.mean(precs),
        'Recall': np.mean(recs),
        'F1-Score': np.mean(f1s),
        'Avg Cost (2*FP+FN)': np.mean(costs),
        'Avg Nodes': np.mean(nodes_list),
        'Avg Depth': np.mean(depths_list),
        'Time (ms)': np.mean(times_ms),
    })

df_dtl_mixed = pd.DataFrame(dtl_mixed_records)
display(df_dtl_mixed.round(4))

# 2. Komparasi Head-to-Head From Scratch vs scikit-learn (Parameter Wajib FOUNDATION.md Bagian 4.4)
sklearn_comp_configs = [
    ('Scratch ID3 (criterion=id3)', lambda: DecisionTree(criterion='id3', use_pruning=False)),
    ('Sklearn ID3 (criterion=entropy)', lambda: get_sklearn_dtl('id3')),
    ('Scratch C4.5 (criterion=c45 + pruning)', lambda: DecisionTree(criterion='c45', use_pruning=True)),
    ('Sklearn C4.5 (criterion=entropy, ccp_alpha=0.01)', lambda: get_sklearn_dtl('c45')),
    ('Scratch CART (criterion=cart)', lambda: DecisionTree(criterion='cart', use_pruning=False)),
    ('Sklearn CART (criterion=gini)', lambda: get_sklearn_dtl('cart')),
]

comp_records = []
for label, builder in sklearn_comp_configs:
    accs, precs, recs, f1s, costs, times_ms = [], [], [], [], [], []
    for train_idx, val_idx in k_fold_split(X_dummy, y_dummy, k=5, seed=RANDOM_SEED):
        t0 = time.perf_counter()
        clf = builder()
        clf.fit(X_dummy[train_idx], y_dummy[train_idx])
        preds = clf.predict(X_dummy[val_idx])
        elapsed = (time.perf_counter() - t0) * 1000.0
        accs.append(accuracy_score(y_dummy[val_idx], preds))
        precs.append(precision_score(y_dummy[val_idx], preds))
        recs.append(recall_score(y_dummy[val_idx], preds))
        f1s.append(f1_score(y_dummy[val_idx], preds))
        costs.append(calculate_custom_cost(y_dummy[val_idx], preds))
        times_ms.append(elapsed)
    comp_records.append({
        'Model Comparison (5-Fold CV)': label,
        'Accuracy': np.mean(accs),
        'Precision': np.mean(precs),
        'Recall': np.mean(recs),
        'F1-Score': np.mean(f1s),
        'Avg Cost': np.mean(costs),
        'Time (ms)': np.mean(times_ms),
    })

df_dtl_comp = pd.DataFrame(comp_records)
display(df_dtl_comp.round(4))

# 3. Simpan Model Terbaik & Ekspor Visualisasi Top-N Pohon Keputusan (+1 Poin Bonus)
best_dtl_model = DecisionTree(criterion='c45', use_pruning=True, val_split_ratio=0.2, random_state=RANDOM_SEED)
best_dtl_model.feature_names_ = dtl_feat_names
best_dtl_model.fit(X_dtl_eval, y_dtl_eval)

model_out_path = os.path.join(root_dir, 'output', 'models', 'dtl_model.pkl')
plot_out_path = os.path.join(root_dir, 'output', 'plots', 'dtl_tree_top_n.png')
best_dtl_model.save(model_out_path)
export_tree_visualization(best_dtl_model, top_n=3, feature_names=dtl_feat_names, output_path=plot_out_path)

## 3. Logistic Regression — Anggota 3
Komparasi `src.logistic_regression.LogisticRegression` ($L_1, L_2$, Elastic Net + Adam) vs `sklearn.linear_model.LogisticRegression` & Video Lintasan Loss

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLogReg
# TODO (Anggota 3): Evaluasi LogisticRegression from scratch vs scikit-learn & render video kontur

## 4. Support Vector Machine (SVM) — Anggota 4
Komparasi `src.svm.SVM` (Linear, Polynomial, Gaussian RBF) vs `sklearn.svm.SVC` & Plot Bidang Pemisah 2D

In [ ]:
from sklearn.svm import SVC
# TODO (Anggota 4): Evaluasi SVM from scratch vs scikit-learn & simpan 3 plot margin 2D

## 5. Naive Bayes & TAN — Anggota 5
Komparasi `src.naive_bayes.NaiveBayes` vs `sklearn.naive_bayes.GaussianNB` & Visualisasi Graf DAG Bayesian Network

In [ ]:
from sklearn.naive_bayes import GaussianNB
# TODO (Anggota 5): Evaluasi NaiveBayes / TAN from scratch vs scikit-learn & simpan graf BN